# 07 — Cannibalization Robustness

**Question.** Within the same brand and store, do other-size price changes shift the focal size's demand? If yes, the optimizer's per-cell raise-price recommendation over-states portfolio profit by ignoring within-brand substitution.

**Approach.**

1. Build a per-row, baseline-quantity-weighted **same-brand other-size price index** `log_p_same`. Baseline weights are time-invariant within `(brand, store)` so the index does not absorb current-week demand shocks (the standard endogeneity worry with current-share weights).
2. Fit three nested fixed-effect models on the **common sample** (rows where the same-brand index is defined) so the coefficients are comparable:

   - **M0** baseline: `log_q ~ log_p + promo`
   - **M1** + cross-brand index: adds `log_comp_price`
   - **M2** + same-brand other-size: adds `log_p_same`

   FE = brand × size × store + week. Cluster-robust SEs at brand × size × store.
3. Re-fit M2 by top-5 brand to check β_same is not driven by one big brand.
4. Plug β_same into an **ex-post spillover adjustment** for the top-10 portfolio candidates.
5. Trigger one of three actions per the project decision rule:

   | β_same outcome | top-10 |Δ%| | action |
   |---|---|---|
   | positive & sig & median \|adj\| > 15% | material | rebuild optimizer with joint objective |
   | positive & sig & 10–15% | borderline | sensitivity range only |
   | small / unstable / wrong-sign | limited | diagnostic warning, optimizer untouched |

**Critical design choice.** This notebook is a **diagnostic**, not a refactor. We do **not** plug β_same into the optimizer unless the magnitude justifies it. Joint optimization is a much bigger architectural change (recommendations become bundles, not cells), and we only pay that cost if the data demand it.

## §1 — Setup and load the demand modeling dataset

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
PROCESSED    = PROJECT_ROOT / 'data' / 'processed'
REPORTS      = PROJECT_ROOT / 'reports'
FIGURES      = REPORTS / 'figures'

mdl = pd.read_parquet(PROCESSED / 'demand_modeling_dataset.parquet').copy()
mdl['WEEK_cat'] = mdl['WEEK'].astype('category')

print(f'rows={len(mdl):,}  brands={mdl["brand_final"].nunique()}  '
      f'stores={mdl["STORE"].nunique()}  weeks={mdl["WEEK"].nunique()}')

sizes_per_bs = mdl.groupby(['brand_final','STORE'], observed=True)['size_oz_rounded'].nunique()
print(f'brand-store combos: {len(sizes_per_bs):,};  '
      f'with ≥2 sizes: {(sizes_per_bs >= 2).sum():,} '
      f'({(sizes_per_bs >= 2).mean()*100:.1f}%)')

Every brand-store has ≥2 sizes (in fact ≥3) so the same-brand index is well-defined essentially everywhere.

## §2 — Build the same-brand other-size price index

For each row $(b,k,s,t)$ where $k$ is the focal size:

$$
\log P^{\text{same}}_{bkst}
\;=\;
\frac{\sum_{k'\neq k}\, w_{bsk'} \cdot \log P_{bk'st}}
     {\sum_{k'\neq k}\, w_{bsk'}},
\qquad
w_{bsk'} = \overline{Q}_{bsk'}.
$$

**Why baseline weights, not current-week shares.** Current-week shares are mechanically negatively correlated with current-week prices (cheap = sells more = bigger weight), so a current-share index would inject endogenous variation into a regressor that is supposed to *control* for endogeneity. Baseline weights are computed once per `(brand, store, size)` from the full-sample mean quantity and held fixed. The remaining within-week variation in `log_p_same` then comes entirely from rivals' price moves at constant baseline composition — what we want.

In [ ]:
# A) baseline weights per (brand, store, size)
bsk_qty = (mdl.groupby(['brand_final','STORE','size_oz_rounded'], observed=True)
              ['quantity'].mean().rename('baseline_q').reset_index())
print(f'baseline weights for {len(bsk_qty):,} (brand,store,size) cells')

# B) per-row same-brand index. Compute Σ(log_p · w) and Σw across ALL sizes in
#    the brand-store-week, then subtract the row's own contribution.
work = mdl.merge(bsk_qty, on=['brand_final','STORE','size_oz_rounded'], how='left')
work['logp_w'] = work['log_p'] * work['baseline_q']
agg = (work.groupby(['brand_final','STORE','WEEK'], observed=True)
            .agg(sum_logp_w=('logp_w','sum'),
                 sum_w=('baseline_q','sum'),
                 n_sizes_in_week=('size_oz_rounded','nunique'))
            .reset_index())
work = work.merge(agg, on=['brand_final','STORE','WEEK'], how='left')

other_w   = work['sum_w']      - work['baseline_q']
other_lpw = work['sum_logp_w'] - work['logp_w']
work['log_p_same']           = np.where(other_w > 0, other_lpw / other_w, np.nan)
work['n_other_sizes_in_week']= work['n_sizes_in_week'] - 1
work['same_brand_defined']   = work['log_p_same'].notna()

mdl['log_p_same']            = work['log_p_same'].values
mdl['n_other_sizes_in_week'] = work['n_other_sizes_in_week'].values
mdl['same_brand_defined']    = work['same_brand_defined'].values

n_def = int(mdl['same_brand_defined'].sum())
print(f'rows with same-brand index defined: {n_def:,} of {len(mdl):,} '
      f'({n_def/len(mdl)*100:.1f}%)')
print(f'median # other sizes per brand-store-week: '
      f'{int(mdl.loc[mdl["same_brand_defined"],"n_other_sizes_in_week"].median())}')

In [ ]:
mdl_def = mdl.loc[mdl['same_brand_defined']].copy()
corr = float(mdl_def[['log_p', 'log_p_same']].corr().iloc[0, 1])
print(f'corr(log_p_focal, log_p_same) = {corr:+.3f}')

The two log-prices are correlated (both driven by category-wide price drift, common cost shocks, and store-level pricing styles) but far from collinear — there is meaningful within-week, within-brand variation for the regression to identify β_same off of.

## §3 — Three FE models on the common sample

All three models share the same sample (rows where `log_p_same` is defined) so the coefficient changes from M0 → M1 → M2 are not driven by sample composition. FE = `brand × size × store` + `week`. SEs clustered at brand × size × store.

In [ ]:
from linearmodels.iv.absorbing import AbsorbingLS

def fit_absorbing(data: pd.DataFrame, y: str, X: list[str],
                  absorb_cols: list[str]):
    dep  = data[y].astype('float64')
    exog = data[X].astype('float64').assign(const=1.0)[['const', *X]]
    absorb = data[absorb_cols]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        warnings.simplefilter('ignore', RuntimeWarning)
        warnings.simplefilter('ignore', UserWarning)
        mod = AbsorbingLS(dep, exog, absorb=absorb)
        res = mod.fit(cov_type='clustered', clusters=data[absorb_cols[0]])
    return res

mdl_def['WEEK_cat'] = mdl_def['WEEK'].astype('category')

specs = {
    'M0_baseline':        ['log_p', 'promo_any_int'],
    'M1_with_cross':      ['log_p', 'log_comp_price', 'promo_any_int'],
    'M2_with_same_brand': ['log_p', 'log_comp_price', 'log_p_same', 'promo_any_int'],
}
results = {}
for name, X in specs.items():
    print(f'fitting {name} ...')
    results[name] = fit_absorbing(mdl_def, y='log_q', X=X,
                                  absorb_cols=['brand_size_store', 'WEEK_cat'])

In [ ]:
def fmt(res, key):
    if key not in res.params.index: return '—'
    b, se = res.params[key], res.std_errors[key]
    star = '***' if abs(b/se) > 2.58 else ('**' if abs(b/se) > 1.96 else
            ('*' if abs(b/se) > 1.65 else ''))
    return f'{b:+.3f}{star} ({se:.3f})'

print(f'N obs (common sample): {int(results["M0_baseline"].nobs):,}\n')
print(f'{"spec":<22s}{"β_own":>14s}{"β_cross":>14s}{"β_same":>14s}{"θ_promo":>14s}{"R²":>9s}')
print('-' * 87)
for name, res in results.items():
    print(f'{name:<22s}{fmt(res,"log_p"):>14s}{fmt(res,"log_comp_price"):>14s}'
          f'{fmt(res,"log_p_same"):>14s}{fmt(res,"promo_any_int"):>14s}'
          f'{res.rsquared:>9.4f}')

**Read of the table.**

- β_own becomes *more* negative when we add `log_p_same` (M0: −1.75 → M2: −1.79). That is the classic sign of mild upward bias in own-price elasticity when same-brand prices are omitted (focal and sister-size prices tend to move together at promo time, masking some of the own-price response).
- β_same is positive and highly significant. Substitution exists.
- The R² gain from adding β_same is small (0.7482 → 0.7484). This is the first hint that even though β_same is detectable, it does not move the dependent variable much — most of `log_q` variance is already absorbed by FE + own price.

In [ ]:
# Persist the coefficient table.
coef_rows = []
for name, res in results.items():
    for k in ['log_p','log_comp_price','log_p_same','promo_any_int']:
        if k in res.params.index:
            coef_rows.append({
                'spec': name, 'term': k,
                'coef': float(res.params[k]),
                'se':   float(res.std_errors[k]),
                'tstat':  float(res.tstats[k]),
                'pvalue': float(res.pvalues[k]),
                'n_obs':    int(res.nobs),
                'rsquared': float(res.rsquared),
            })
coef_df = pd.DataFrame(coef_rows)
coef_df.to_csv(PROCESSED / 'cannibalization_model_coefficients.csv', index=False)
coef_df

## §4 — Top-brand sub-samples (M3)

If β_same in the pooled fit were driven by one or two big brands, the global value would be misleading for the rest. Re-fit M2 separately for each of the top-5 brands by row count.

In [ ]:
top_brands = (mdl_def.groupby('brand_final', observed=True).size()
              .sort_values(ascending=False).head(5).index.tolist())
print('top 5 brands:', top_brands, '\n')

brand_rows = []
for b in top_brands:
    sub = mdl_def.loc[mdl_def['brand_final'] == b].copy()
    if len(sub) < 5_000 or sub['STORE'].nunique() < 2:
        continue
    sub['WEEK_cat'] = sub['WEEK'].astype('category')
    res_b = fit_absorbing(sub, y='log_q',
                          X=['log_p','log_comp_price','log_p_same','promo_any_int'],
                          absorb_cols=['brand_size_store','WEEK_cat'])
    print(f'{b:<25s} N={int(res_b.nobs):>8,}  '
          f'β_own={res_b.params["log_p"]:+.3f}({res_b.std_errors["log_p"]:.3f})  '
          f'β_same={res_b.params["log_p_same"]:+.3f}({res_b.std_errors["log_p_same"]:.3f})')
    brand_rows.append({'brand': b, 'n_obs': int(res_b.nobs),
                       'beta_own':  float(res_b.params['log_p']),
                       'se_own':    float(res_b.std_errors['log_p']),
                       'beta_same': float(res_b.params['log_p_same']),
                       'se_same':   float(res_b.std_errors['log_p_same'])})
brand_df = pd.DataFrame(brand_rows)

Four of the top five brands show positive significant β_same in roughly the same range (+0.20 to +0.51); only Private Label is small and insignificant. The pooled result is not an artifact of one brand — substitution is broad-based among national brands.

## §5 — Ex-post spillover adjustment for top-10 candidates

For each portfolio candidate $K$ at $(b, s)$ with $\Delta\log P_K = \log(P^{\text{opt}}_K/\bar P_K)$, the model implies sister size $K'$'s log-quantity changes by

$$
\Delta\log Q_{K'} \;=\; \beta_{\text{same}} \cdot w_{K \to K'} \cdot \Delta\log P_K,
$$

where $w_{K \to K'} = \bar Q_K \,/\, (\sum_{j} \bar Q_j - \bar Q_{K'})$ is the focal's weight inside $K'$'s same-brand-other-size index (denominator excludes $K'$ itself by construction). The spillover profit is then $\sum_{K'} m_{K'} \cdot \bar Q_{K'} \cdot (e^{\Delta\log Q_{K'}} - 1)$ with $m_{K'} = \bar P_{K'} - \bar c_{K'}$.

This is the **portfolio profit correction** the optimizer would silently miss under independent-cell optimization.

In [ ]:
top10 = pd.read_csv(PROCESSED / 'top_recommendations_diverse.csv')
cells = pd.read_parquet(PROCESSED / 'cell_baselines.parquet')

beta_same_M2 = float(results['M2_with_same_brand'].params['log_p_same'])
print(f'using β_same = {beta_same_M2:+.4f} from M2_with_same_brand\n')

adj_rows = []
for _, focal in top10.iterrows():
    b = focal['brand_final']
    k = float(focal['size_oz_rounded'])
    s = int(focal['STORE'])

    sisters = cells.loc[(cells['brand_final'] == b) &
                        (cells['STORE'] == s) &
                        (cells['size_oz_rounded'] != k)].copy()

    delta_logp_focal = np.log(focal['opt_price'] / focal['mean_p'])
    bsk_here = bsk_qty.loc[(bsk_qty['brand_final'] == b) & (bsk_qty['STORE'] == s)]
    base_q_focal = float(bsk_here.loc[bsk_here['size_oz_rounded'] == k, 'baseline_q'].iloc[0])
    base_q_total = float(bsk_here['baseline_q'].sum())

    spillover = 0.0
    for _, sis in sisters.iterrows():
        bq_sister = float(bsk_here.loc[bsk_here['size_oz_rounded'] == sis['size_oz_rounded'],
                                       'baseline_q'].iloc[0])
        denom = base_q_total - bq_sister
        if denom <= 0:
            continue
        w = base_q_focal / denom
        d_logq_sis = beta_same_M2 * w * delta_logp_focal
        d_q_sis    = float(sis['mean_q']) * (np.exp(d_logq_sis) - 1.0)
        margin     = float(sis['mean_p']) - float(sis['mean_cost'])
        spillover += d_q_sis * margin

    adjusted = float(focal['profit_lift_abs']) + spillover
    adj_pct  = (adjusted - focal['profit_lift_abs']) / max(abs(focal['profit_lift_abs']), 1e-6) * 100
    adj_rows.append({
        'brand_final':       focal['brand_final'],
        'size_oz_rounded':   focal['size_oz_rounded'],
        'STORE':             focal['STORE'],
        'mean_p':            focal['mean_p'],
        'opt_price':         focal['opt_price'],
        'profit_lift_abs':   focal['profit_lift_abs'],
        'n_sisters':         len(sisters),
        'spillover_profit':  spillover,
        'adjusted_lift_abs': adjusted,
        'adjustment_pct':    adj_pct,
    })

adj_df = pd.DataFrame(adj_rows)
adj_df

In [ ]:
med_pct = float(adj_df['adjustment_pct'].abs().median())
max_pct = float(adj_df['adjustment_pct'].abs().max())
print(f'median |adjustment| across top-10 = {med_pct:.1f}%')
print(f'max    |adjustment| across top-10 = {max_pct:.1f}%')

Spillover is **uniformly positive** (raising focal price moves a few units to sister sizes, which is profit-positive at sister-size margins) and **small in magnitude** — under 5% even at the worst case. This is well below the 10–15% band that would warrant a joint-optimizer rebuild.

Why so small? Two reasons compound:

1. β_same ≈ 0.23 is much smaller than |β_own| ≈ 1.79. Substitution is real but inelastic compared to the own-price response.
2. Each top-10 cell is one of ~20+ sizes at its (brand, store), so the focal's weight $w_{K 	o K'}$ in any single sister's index is small (typically a few percent).

## §6 — Decision rule (per project spec)

In [ ]:
beta_same_t = float(results['M2_with_same_brand'].tstats['log_p_same'])
beta_same_p = float(results['M2_with_same_brand'].pvalues['log_p_same'])
sig = abs(beta_same_t) > 1.96
positive = beta_same_M2 > 0

if sig and positive and med_pct > 15:
    decision = ('**Material**: β_same is positive, statistically significant, and '
                f'shifts top-10 lifts by a median {med_pct:.1f}%. The optimizer '
                'should be revised to include same-brand spillover-adjusted profit '
                'before the next iteration of recommendations.')
elif sig and positive and med_pct >= 10:
    decision = ('**Borderline**: β_same is positive and significant, with '
                f'top-10 adjustment {med_pct:.1f}% (10–15% band). Surface as a '
                'sensitivity range in the optimizer and flag affected candidates; '
                'do not rebuild the optimizer yet.')
elif sig and positive:
    decision = ('**Limited first-order impact**: β_same is positive and significant '
                f'but small in magnitude — top-10 adjustment is {med_pct:.1f}% (median, '
                f'max {max_pct:.1f}%). Document the diagnostic; the MVP optimizer is '
                'fit-for-purpose without joint optimization.')
elif not positive:
    decision = (f'**Wrong-signed**: β_same = {beta_same_M2:+.3f} (p = {beta_same_p:.3f}) is '
                'negative or zero, inconsistent with substitution. Likely contamination '
                'by common brand demand shocks or correlated promo pricing. Do **not** '
                'plug β_same into the optimizer — keep as a diagnostic warning.')
else:
    decision = (f'**Inconclusive**: β_same = {beta_same_M2:+.3f} (p = {beta_same_p:.3f}) is '
                'not statistically distinguishable from zero. Treat as evidence that '
                'within-brand cannibalization is small or unidentified at this grain.')

print(decision)

## §7 — Persist diagnostics, figure, summary

In [ ]:
diag = mdl[['brand_final','size_oz_rounded','STORE','WEEK','log_p','log_p_same',
            'n_other_sizes_in_week','same_brand_defined']].copy()
diag.to_parquet(PROCESSED / 'cannibalization_diagnostics.parquet', index=False)
print(f'wrote {PROCESSED}/cannibalization_diagnostics.parquet  ({len(diag):,} rows)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = ['β_own (log_p)', 'β_cross (other brands)',
          'β_same (own brand, other size)', 'θ_promo']
plot_specs = [('M1_with_cross', '#888'), ('M2_with_same_brand', '#3a78b8')]

x = np.arange(len(labels)); width = 0.35
for i, (spec, color) in enumerate(plot_specs):
    res = results[spec]
    b  = [float(res.params[t])      if t in res.params.index else np.nan
          for t in ['log_p','log_comp_price','log_p_same','promo_any_int']]
    se = [float(res.std_errors[t])  if t in res.params.index else 0.0
          for t in ['log_p','log_comp_price','log_p_same','promo_any_int']]
    pos = x + (i - 0.5) * width
    ax.bar(pos, b, width=width, color=color,
           label=spec.replace('_', ' '),
           yerr=[1.96 * s for s in se], capsize=3)

ax.axhline(0, lw=0.6, color='k')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=12, ha='right')
ax.set_ylabel('coefficient')
ax.set_title('Cannibalization robustness — same-brand other-size price index (95% CI)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES / 'cannibalization_coefficients.png', dpi=140)
plt.show()

## §8 — Write the cross-doc summary

`reports/cannibalization_robustness_summary.md` is rendered into Page 5 (Limitations) so the diagnostic stays linked to the user-facing claim.

In [ ]:
res2 = results['M2_with_same_brand']
out_path = REPORTS / 'cannibalization_robustness_summary.md'

lines = []
def w(s=''): lines.append(s)

w('# Cannibalization Robustness Summary')
w(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
w('')
w(f'**Headline.** β_same (own-brand other-size cross-price coef) = '
  f'{beta_same_M2:+.3f} (SE {res2.std_errors["log_p_same"]:.3f}, '
  f'p = {beta_same_p:.3f}). Median |spillover adjustment| across top-10 candidates = '
  f'{med_pct:.1f}% (max {max_pct:.1f}%).')
w('')
w(decision)
w('')
w('---')
w('')
w('## Question')
w('Within the same brand and store, do other-size price changes shift the focal '
  "size's demand? If yes, the optimizer's per-cell raise-price recommendation "
  'over-states portfolio profit by ignoring within-brand substitution.')
w('')
w('## Coefficient table (common sample)')
w(f'- N obs = {int(res2.nobs):,}')
w('')
w('| spec | β_own | β_cross | β_same | θ_promo | R² |')
w('|------|------:|------:|------:|------:|------:|')
for name, res in results.items():
    w(f'| {name} | {fmt(res,"log_p")} | {fmt(res,"log_comp_price")} | '
      f'{fmt(res,"log_p_same")} | {fmt(res,"promo_any_int")} | {res.rsquared:.4f} |')
w('')
w('Stars: * |t|>1.65, ** |t|>1.96, *** |t|>2.58. Cluster-robust SE, clustered at brand-size-store.')
w('')
w('## Top-brand check (M3)')
for _, r in brand_df.iterrows():
    w(f'- {r["brand"]}: N={r["n_obs"]:,}, β_own={r["beta_own"]:+.3f} ({r["se_own"]:.3f}), '
      f'β_same={r["beta_same"]:+.3f} ({r["se_same"]:.3f})')
w('')
w('## Spillover-adjusted top-10')
w('| brand | size | store | focal Δπ | sisters | spillover | adjusted Δπ | Δ% |')
w('|---|---:|---:|---:|---:|---:|---:|---:|')
for _, r in adj_df.iterrows():
    w(f"| {r['brand_final']} | {r['size_oz_rounded']:.2f} | {int(r['STORE'])} | "
      f"${r['profit_lift_abs']:.0f} | {int(r['n_sisters'])} | "
      f"${r['spillover_profit']:+.0f} | ${r['adjusted_lift_abs']:.0f} | "
      f"{r['adjustment_pct']:+.1f}% |")
w('')
w(f'**Median |adjustment| = {med_pct:.1f}%; max = {max_pct:.1f}%.**')
w('')
w('## Artifacts')
w('- `data/processed/cannibalization_diagnostics.parquet` — per-row same-brand index')
w('- `data/processed/cannibalization_model_coefficients.csv` — M0/M1/M2 coefficients')
w('- `reports/figures/cannibalization_coefficients.png` — coefficient bar chart')
w('- This file (`reports/cannibalization_robustness_summary.md`)')

out_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'wrote {out_path}  ({out_path.stat().st_size:,} bytes)')

## §9 — Takeaways

1. **Within-brand substitution exists** (β_same = +0.231***), broad-based across the top national brands.
2. **It is not large enough to change top-10 portfolio recommendations** — median adjustment 2.5%, max 4.0%, well under the 10–15% band that would justify rebuilding the optimizer to a joint objective.
3. **Direction is profit-positive**: the focal candidate's raise-price move shifts a small amount of demand to sister sizes, which adds (rather than subtracts) profit at sister-size margins.
4. **Decision**: keep the per-cell optimizer; surface this diagnostic in the Limitations page so users know it has been quantitatively checked, not just acknowledged in passing.
5. **What would change this**: if β_same were closer to ~0.6, or if the typical (brand, store) had only 2–3 sizes (concentrating $w_{K \to K'}$), top-10 adjustments could exceed 15% and a joint-optimization rebuild would be needed. Re-run this notebook on a category with fewer SKUs per shelf.